In [ ]:
# ============================================================
# Спектрограммы (STFT, ЛИНЕЙНАЯ частота, амплитуда в дБ)
# ============================================================
import os
import re
from pathlib import Path
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

# ---------- НАСТРОЙКИ ----------
# Каждый элемент: (путь_к_файлу, [(начало, конец), (начало, конец), ...])
# Время можно задавать либо числом (секунды), либо строкой "mm:ss" / "hh:mm:ss"
TASKS = [
    ("output/input/input_30min_16k.wav", [(0, 30)]), # 16 kHz, исходный микс (30 мин)
    ("output/tse_out/usef_tfgridnet_8k.wav", [(0, 30)]),
    ("output/tse_out/usef_tfgridnet_16k.wav", [(0, 30)]),
    ("output/tse_out/solospeech_30min.wav", [(0, 30)]),
    ("output/tse_out/posneg_proposed.wav", [(0, 30)]),
    ("output/tse_out/posneg_improved.wav", [(0, 30)]),
    ("output/tse_out/meanflow.wav", [(0, 30)]),
    ("output/tse_out/tselm.wav", [(0, 30)]),
]

OUT_DIR = Path("spectrograms")   # куда сохранять PNG
N_FFT = 2048                     # размер окна БПФ
HOP_LENGTH = 512                 # шаг между окнами
WINDOW = "hann"
DPI = 150
# --------------------------------


def parse_time(t) -> float:
    """Принимает число (сек) или строку 'mm:ss' / 'hh:mm:ss' → секунды (float)."""
    if isinstance(t, (int, float)):
        return float(t)
    if isinstance(t, str):
        parts = t.strip().split(":")
        parts = [float(p) for p in parts]
        if len(parts) == 2:           # mm:ss
            m, s = parts
            return m * 60 + s
        if len(parts) == 3:           # hh:mm:ss
            h, m, s = parts
            return h * 3600 + m * 60 + s
    raise ValueError(f"Не удалось распознать время: {t!r}")


def latex_safe(name: str) -> str:
    """Имя файла, безопасное для \\includegraphics: только [a-z0-9-], без точек и '_'."""
    name = name.lower().replace("_", "-")
    name = re.sub(r"[^a-z0-9-]+", "-", name)
    name = re.sub(r"-+", "-", name).strip("-")
    return name or "spec"


def fmt_sec(x: float) -> str:
    """Секунды в имя файла без точки (точка ломает разбор расширения в LaTeX)."""
    return str(int(x)) if float(x).is_integer() else f"{x:.2f}".replace(".", "p")


def make_spectrogram(file_path: str, start, end, out_dir: Path) -> Path:
    """Строит и сохраняет STFT-спектрограмму (ЛИНЕЙНАЯ частота, дБ) для отрезка [start, end]."""
    start_s = parse_time(start)
    end_s = parse_time(end)
    if end_s <= start_s:
        raise ValueError(f"end ({end_s}) должен быть больше start ({start_s})")

    # Загружаем только нужный фрагмент, без ресемплинга
    duration = end_s - start_s
    y, sr = librosa.load(file_path, sr=None, mono=True,
                         offset=start_s, duration=duration)

    if y.size == 0:
        raise RuntimeError(f"Пустой фрагмент {start_s}-{end_s} с в {file_path}")

    # STFT → амплитуда → дБ
    stft = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH, window=WINDOW)
    S_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

    # Рисуем
    fig, ax = plt.subplots(figsize=(12, 5))
    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=HOP_LENGTH,
        x_axis="time",
        y_axis="linear",    # линейная шкала частот
        ax=ax,
        cmap="magma",
    )
    cbar = fig.colorbar(img, ax=ax, format="%+2.0f")
    cbar.set_label("Амплитуда, дБ (отн. макс.)")

    stem = Path(file_path).stem
    ax.set_title(f"{stem}   |   {start_s:.2f}–{end_s:.2f} с   |   sr = {sr} Гц")
    ax.set_xlabel("Время, с (от начала фрагмента)")
    ax.set_ylabel("Частота, Гц (лин. шкала)")

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{latex_safe(stem)}-{fmt_sec(start_s)}-{fmt_sec(end_s)}.png"
    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    return out_path


# ---------- ЗАПУСК ----------
saved = []
for file_path, intervals in TASKS:
    if not os.path.exists(file_path):
        print(f"[!] Файл не найден: {file_path}")
        continue
    for start, end in intervals:
        try:
            p = make_spectrogram(file_path, start, end, OUT_DIR)
            saved.append(p)
            print(f"[+] {p}")
        except Exception as e:
            print(f"[!] Ошибка для {file_path} [{start}, {end}]: {e}")

print(f"\nГотово. Сохранено файлов: {len(saved)}")